# Module 33 — Exercise 3: Task Queue Worker with Idempotency Keys

In distributed systems, networks drop acknowledgements, resulting in duplicate task executions. Workers must be idempotent: executing the same job twice must produce the exact same outcome without duplicating side-effects (e.g., billing twice).

In this exercise, you will implement an Idempotent Task Worker.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 33 README |



# Your turn


### Task 1: Idempotent Payment Processor

Implement `process_payment(store, idempotency_key, account_id, amount)`:
- Check `store` for `idempotency_key`. If already present, return the saved response without re-executing balance deduction.
- If not present, deduct `amount` from `account_id`, store transaction record with `idempotency_key`, and return `{"status": "processed", "amount": amount}`.


In [ ]:
# ANSWER 1
def process_payment(store: dict, accounts: dict, idempotency_key: str, account_id: str, amount: float) -> dict:
    if idempotency_key in store:
        return store[idempotency_key]

    if accounts.get(account_id, 0.0) < amount:
        res = {"status": "failed", "reason": "insufficient_funds"}
    else:
        accounts[account_id] -= amount
        res = {"status": "processed", "amount": amount}

    store[idempotency_key] = res
    return res



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

accounts = {"acc_101": 200.0}
store = {}

# Execute first time
res1 = process_payment(store, accounts, "idem_key_abc", "acc_101", 50.0)
balance_after_1 = accounts["acc_101"]

# Duplicate request with same idempotency key
res2 = process_payment(store, accounts, "idem_key_abc", "acc_101", 50.0)
balance_after_2 = accounts["acc_101"]

results = [
    check(res1["status"] == "processed" and balance_after_1 == 150.0, "Task 1: First payment processed and balance deducted"),
    check(res2["status"] == "processed" and balance_after_2 == 150.0, "Task 1: Duplicate payment returned cached response without double-billing"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

